[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 06](README.md)

# CUDA: coalescencia, memoria compartida y tiling

**Tema:** 06 · **Sesiones:** 27, 28 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo aumentar reutilización sin exceder recursos ni romper bordes y sincronización?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Tiling intercambia cooperación y memoria compartida por reutilización. El beneficio depende de coalescencia, recursos por bloque y tratamiento correcto de bordes.

**Prerrequisitos.**

- C++20, memoria y descomposición por datos.
- Modelo host–dispositivo y medición extremo a extremo.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Relacionar coalescencia, bancos y memoria compartida.
- Calcular recursos de un tile.
- Validar GEMM tiled frente a CPU para dimensiones irregulares.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Tiling carga datos reutilizados en memoria compartida y sincroniza antes de consumirlos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Un tile mayor puede aumentar reutilización pero también registros, memoria compartida y presión de ocupación.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Las dimensiones no múltiplos requieren cargas condicionadas y ceros fuera del dominio.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- SIMT — ejecución de hilos agrupados sobre una instrucción
- warp — grupo de hilos planificado conjuntamente
- coalescencia — agrupación eficiente de accesos contiguos
- tile — bloque de datos reutilizado localmente


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Cuda Grid Tiling

![Jerarquía grid–bloque–hilo y tile compartido](../../images/cuda-grid-tiling.svg)

**Cómo leerlo.** Primero ubica un hilo dentro de su bloque y grid; después observa que la cooperación y sincronización ocurren dentro del bloque que reutiliza el tile.

### Jerarquia Memoria

![Jerarquía de memoria y costo de movimiento](../../images/jerarquia-memoria.svg)

**Cómo leerlo.** Al descender aumenta la capacidad y suele aumentar la latencia. La optimización busca reutilizar datos antes de solicitar un nivel más lejano.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "06"
NOTEBOOK = "06_cuda/02_memoria_tiling.ipynb"
assert (ROOT / "curso" / "notebooks" / "06_cuda" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Recursos por tile

**Situación.** Se calculan hilos, bloques, memoria compartida y tiles de K para GEMM.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
def tile_resources(m, n, k, tile, bytes_per_value=4):
    return {
        "grid": ((n+tile-1)//tile, (m+tile-1)//tile),
        "threads_per_block": tile*tile,
        "shared_bytes": 2*tile*tile*bytes_per_value,
        "k_tiles": (k+tile-1)//tile,
    }
for tile in (8, 16, 32):
    row = tile_resources(1000, 777, 513, tile)
    assert row["threads_per_block"] <= 1024
    print(tile, row)


### Explicación del resultado

La validez geométrica no garantiza buena ocupación; se consulta el límite real y el perfil del kernel.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Intensidad aproximada

**Situación.** Se compara reutilización ideal de una GEMM ingenua y una tiled.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
def intensity(tile, bytes_per_value=4):
    flops = 2 * tile * tile * tile
    bytes_loaded = 2 * tile * tile * bytes_per_value
    return flops / bytes_loaded
for tile in (8, 16, 32): print(tile, f"{intensity(tile):.2f} FLOP/byte por fase ideal")
assert intensity(32) > intensity(8)


### Lectura razonada

El cálculo ideal omite escrituras, cachés, bordes y recargas; sirve para formular la hipótesis de reutilización.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Por qué aumentar el tile puede reducir rendimiento aunque aumente reutilización ideal?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Comparar GEMM ingenua, tiled y cuBLAS con la misma precisión.
2. Incluir dimensiones no divisibles por tile.
3. Perfilar coalescencia, bancos, ocupación y tiempo total.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Sincronizar fuera de una rama que no alcanza todo el bloque.
- Comparar operaciones diferentes.
- Elegir tile solo por tiempo de un caso.


## Criterios de aceptación

- Bordes comprobados con referencia CPU.
- Recursos por bloque dentro de límites.
- Interpretación sustentada en métricas del perfil.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo aumentar reutilización sin exceder recursos ni romper bordes y sincronización?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Guía CUDA](README.md)
- [Fuentes CUDA heredadas](../../../cuda/03_cuda_thread_programming/)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 06](README.md)
